In [47]:
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from atlassian import Jira

model="llama3.1"
llm = ChatOllama(model=model, temperature=0)

In [51]:
from dotenv import load_dotenv
import os
load_dotenv()
try:
    jira = Jira(
    url=os.getenv("url"),
    username=os.getenv("username"),
    password=os.getenv("password"),
    cloud=True)
except Exception as e:
    print(f"Unable to login to jira, Error: {e}")

https://amagiengg.atlassian.net


In [ ]:
import pandas as pd

# def read_csv_and_organize(csvPath:str):
#     df = pd.read_csv(csvPath)
#     cleared_df = df.dropna(subset=['summary'])
#     docs = []
    
#     for index, row in cleared_df.iterrows():
        
#         data = {
#             'summary': row['summary'],
#             'description': row['description'],
#             'assignee': row['assignee'],
#         }
        
#         docs.append(data)

#     return docs

# print(read_csv_and_organize('../csvFiles/jira_task_list.csv'))


def createJiraTaskFromLocalCSVFile(csvPath:str):
    df = pd.read_csv(csvPath)
    cleared_df = df.dropna(subset=['summary']) 
    for index, row in cleared_df.iterrows():
        fields = {'project':{'key':'AN30'},'issuetype': {'name': 'Task'},'summary': row['summary'], 'description':row['description'], 'assignee':{'id':row['assignee']}}
        res=jira.issue_create(fields)
        print(res)
# {'id': '2784859', 'key': 'AN30-6067', 'self': 'asdf'}
    
createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv') 
# key=2784795

def getIssue():
    print(jira.issue(key))

# getIssue()


In [6]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

store = {}

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a highly capable AI assistant tasked with understanding user queries and responding with the appropriate function from the list provided below. Your response must adhere to the following constraints:

            Functions:
                - read_csv_and_organize(pathToCSV)

            Constraints:
                - Only use the functions listed above. Do not generate or suggest any other functions.
                - Ensure that the function you generate directly addresses the query made by the user.
                - If the query does not correspond to any function you are allowed to use, respond with an empty string ''.
                - Include only the function name and arguments in your response, without any additional text.

            Examples:
                - Query: "Can you read the csv in the path '../csvFiles/jira_task_list.csv'" 
                Response: "read_csv_and_organize('../csvFiles/jira_task_list.csv')"
                - Query: "read the csv in the path '../csvFiles/jira_task_list.csv'" 
                Response: "read_csv_and_organize('../csvFiles/jira_task_list.csv')"
                - Query: "get the data from the csv in the path '../csvFiles/jira_task_list.csv'" 
                Response: "read_csv_and_organize('../csvFiles/jira_task_list.csv')"
                - Query: "from the csv in the path '../csvFiles/jira_task_list.csv' get the data" 
                Response: "read_csv_and_organize('../csvFiles/jira_task_list.csv')"
                - Query: "Subtract 30 from 100."
                Response: ''

            Be mindful that only the functions defined above are valid, and the response must match the function signature exactly.
            """
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain =  RunnablePassthrough.assign(messages=itemgetter("messages")) | prompt | llm 

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

config = {"configurable": {"session_id": "new"}}


In [12]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Can you read the csv in the path '../csvFiles/jira_task_list.csv'")]},
    config=config,
)
res = response.content.strip()
print(res)

read_csv_and_organize('../csvFiles/jira_task_list.csv')


In [ ]:

if(len(res)>2):
    try:
        eval(res)
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Unexpected response:", res)

In [29]:
ar=[1,2,3]
print(len(ar))

3
